# Module 5: Optimize With AgentCore Evidence

This notebook turns observed behavior into optimization experiments.

You will learn the AWS pattern for optimization: inspect evidence, run insights, request recommendations, package a treatment candidate, decide which experiment is runnable, and record the release decision. The key discipline is that an optimization suggestion is not a production change until it has experiment evidence.

In [ ]:

from __future__ import annotations

import json
import os
import sys
import time
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "03-production-deployment").is_dir():
            return candidate
    raise FileNotFoundError("Could not find workshop repo root. Open this notebook from the workshop repo.")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
SECTION_DIR = REPO_ROOT / "05-agentcore-optimization"
SECTION03_DIR = REPO_ROOT / "03-production-deployment"
for path in [SECTION_DIR, SECTION03_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from optimization_contract import (  # noqa: E402
    OPTIMIZATION_MANIFEST_PATH,
    PROMOTION_MANIFEST_PATH,
    RELEASE_DECISION_REPORT_PATH,
    blocked_experiment,
    build_bundle_configuration,
    build_optimization_manifest,
    build_promotion_manifest,
    build_release_decision_report,
    build_treatment_behavior,
    build_tool_recommendation_inputs,
    combined_system_prompt,
    derive_insight_trace_window,
    detect_config_bundle_readiness,
    dry_run_target_canary_plan,
    extract_system_prompt_recommendation,
    extract_tool_description_recommendations,
    load_current_behavior,
    load_gateway_tool_descriptions,
    load_upstream_evidence,
    observed_tool_names,
    log_group_arns,
    recommendation_safe_system_prompt,
    save_json,
    validate_insights_gate,
    validate_recommendation_gate,
    summarize_configuration_bundle_response,
)
from ab_experiment import (  # noqa: E402
    ABExperimentError,
    build_ab_metric_review,
    create_ab_online_evaluation_config,
    create_config_bundle_ab_test,
    create_runtime_gateway_and_target,
    derive_config_bundle_ab_status,
    ensure_ab_gateway_role,
    ensure_gateway_trace_delivery,
    ensure_online_evaluation_alignment,
    finalize_ab_test_for_decision,
    invoke_ab_gateway,
    poll_ab_results,
    require_ab_trace_evidence,
    require_ab_running,
    require_gateway_trace_delivery_ready,
    require_gateway_traffic_success,
    require_online_evaluation_ready,
    stop_ab_test,
    summarize_ab_test,
    summarize_online_evaluation_config,
    wait_for_ab_running,
    wait_for_ab_trace_evidence,
)
from utils import get_user_token, invoke_agent_runtime, redact_email  # noqa: E402

STARTED_AT = time.time()
RUN_SUFFIX = uuid.uuid4().hex[:8]
LIVE_AWS = os.environ.get("SECTION05_SKIP_LIVE_AWS", "0") != "1"
print(f"Repo root: {REPO_ROOT}")
print(f"Section 05 run suffix: {RUN_SUFFIX}")
print(f"Live AWS calls enabled: {LIVE_AWS}")


## Step 1: Load Evidence Inputs

This cell loads the evidence produced by deployment and online observation.

You should learn what information an optimization step needs before it can act: runtime identity, dataset versions, batch evaluation baseline, online evaluation config, feedback candidates, reviewed dataset updates, log groups, service names, and current behavior source hashes. The output table is your checklist that the optimization loop has enough context. The printed service names separate the evidence planes: Insights analyzes the service name present in the Section 04 trace window, while fresh recommendation traffic and A/B scoring use the current runtime endpoint service name.

If this cell fails, the usual cause is missing or stale upstream evidence. Section 03, Section 04 online evidence, and the dataset update must describe the same deployment and dataset lineage. The right next action is to rerun the stale upstream section instead of mixing old production traces with a new runtime deployment.


In [ ]:

evidence = load_upstream_evidence()
behavior = load_current_behavior()

deployment = evidence["deployment"]
dataset = evidence["dataset"]
batch_baseline = evidence["batch_evaluation"]
online_evidence = evidence["online_evidence"]
dataset_update = evidence["dataset_update"]
feedback_candidates = evidence["production_feedback_candidates"]
promoted_feedback = evidence["promoted_feedback_examples"]

REGION = deployment.get("region") or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
ACCOUNT_ID = deployment.get("account_id")
RUNTIME = deployment.get("runtime", {})
RUNTIME_ARN = RUNTIME.get("runtime_arn")
GATEWAY_ARN = deployment.get("gateway", {}).get("gateway_arn")
LOG_GROUP_NAMES = online_evidence.get("log_groups") or ["aws/spans"]
OTEL_SERVICE_NAME = deployment.get("otel_service_name")
ENDPOINT_SERVICE_NAME = f"{RUNTIME.get('runtime_name')}.DEFAULT" if RUNTIME.get("runtime_name") else None
SERVICE_NAMES = [ENDPOINT_SERVICE_NAME] if ENDPOINT_SERVICE_NAME else [name for name in [OTEL_SERVICE_NAME] if name]
# Insights/batch jobs must filter on the same service name the runtime emits in
# its spans (runtime_name + ".DEFAULT"), NOT the ECR repository name. An env
# override is kept for non-standard deployments.
SECTION04_TRACE_SERVICE_NAME = (
    os.environ.get("SECTION05_SECTION04_TRACE_SERVICE_NAME")
    or (SERVICE_NAMES[0] if SERVICE_NAMES else None)
)
BATCH_SERVICE_NAMES = [SECTION04_TRACE_SERVICE_NAME] if SECTION04_TRACE_SERVICE_NAME else SERVICE_NAMES[:1]
RECOMMENDATION_SERVICE_NAMES = SERVICE_NAMES[:1]
INSIGHT_LOG_GROUP_NAMES = [
    name for name in LOG_GROUP_NAMES
    if "/aws/bedrock-agentcore/runtimes/" in str(name)
]
INSIGHT_LOG_GROUP_NAMES = INSIGHT_LOG_GROUP_NAMES or LOG_GROUP_NAMES[:1]
LOG_GROUP_ARNS = log_group_arns(region=REGION, account_id=ACCOUNT_ID, log_group_names=LOG_GROUP_NAMES)

promoted_runtime_error_examples = []
for example in promoted_feedback.get("managed_dataset_examples", []):
    turn_text = " ".join(str(turn.get("expected_response", "")) for turn in example.get("turns", []))
    if "ParamValidationError" in turn_text or "runtimeSessionId" in turn_text:
        promoted_runtime_error_examples.append(example.get("scenario_id"))

feedback_signal_summary = {
    "promoted_runtime_error_example_count": len(promoted_runtime_error_examples),
    "promoted_runtime_error_scenario_ids": promoted_runtime_error_examples,
    "interpretation": "runtime invocation errors feed target-canary/runtime investigation, not prompt/tool-description optimization proof",
}

summary_rows = [
    {"evidence": "deployment", "value": deployment.get("deployment_id")},
    {"evidence": "runtime", "value": RUNTIME.get("runtime_id")},
    {"evidence": "batch baseline", "value": batch_baseline.get("batch_evaluation", {}).get("batch_evaluation_id")},
    {"evidence": "batch status", "value": batch_baseline.get("batch_evaluation", {}).get("status")},
    {"evidence": "dataset baseline version", "value": dataset.get("managed_datasets", {}).get("predefined", {}).get("baseline_dataset_version")},
    {"evidence": "dataset updated version", "value": dataset_update.get("updated_dataset_version")},
    {"evidence": "feedback candidates", "value": feedback_candidates.get("candidate_count")},
    {"evidence": "promoted feedback examples", "value": promoted_feedback.get("promoted_count")},
    {"evidence": "promoted runtime-error examples", "value": feedback_signal_summary["promoted_runtime_error_example_count"]},
    {"evidence": "online evaluation config", "value": online_evidence.get("online_evaluation", {}).get("online_evaluation_config_id")},
]
display(pd.DataFrame(summary_rows))
print(f"Log groups: {LOG_GROUP_NAMES}")
print(f"Insight log groups: {INSIGHT_LOG_GROUP_NAMES}")
print(f"Service names: {SERVICE_NAMES}")
print(f"Batch/optimization service name: {BATCH_SERVICE_NAMES}")
print(f"Endpoint evaluation service name: {ENDPOINT_SERVICE_NAME}")
print(f"Section 04 trace service name: {SECTION04_TRACE_SERVICE_NAME}")
print(f"Recommendation service name: {RECOMMENDATION_SERVICE_NAMES}")


## Step 2: Run AgentCore Insights

This cell asks AgentCore to analyze the Section 04 monitored trace evidence.

Insights summarize patterns across sessions, such as user intents, execution behavior, and failure clusters when available. The trace window is bounded around the Section 04 run so the job analyzes the workshop evidence instead of older test traffic in the account. The managed Insights job uses the runtime application log group, while the broader log-group list remains in the evidence manifest for human trace inspection. Inspect the insight job status, trace window, requested insight IDs, session counts, and error details. A clean result is `COMPLETED`, zero failed sessions, no error details, and the expected user-intent and execution-summary outputs. Failure analysis may be empty when this bounded evidence window has no failure clusters. A partial or failed result is diagnostic evidence, but it should be fixed or explicitly investigated before recommendations are requested.


In [ ]:

dp = boto3.client("bedrock-agentcore", region_name=REGION)
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)

INSIGHT_IDS = [
    "Builtin.Insight.FailureAnalysis",
    "Builtin.Insight.UserIntent",
    "Builtin.Insight.ExecutionSummary",
]
TERMINAL_BATCH_STATUSES = {"COMPLETED", "FAILED", "STOPPED", "COMPLETED_WITH_ERRORS"}
insight_window = derive_insight_trace_window(online_evidence)
start_time = insight_window["start_time"]
end_time = insight_window["end_time"]

insights_result = {
    "status": "SKIPPED" if not LIVE_AWS else "NOT_STARTED",
    "requested_insights": INSIGHT_IDS,
    "trace_window_source": "section04_monitored_sessions_window",
    "trace_window": {
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
    },
    "log_groups": INSIGHT_LOG_GROUP_NAMES,
    "source_log_groups": LOG_GROUP_NAMES,
    "service_names": BATCH_SERVICE_NAMES,
    "monitored_session_count": len(online_evidence.get("monitored_sessions") or []),
}

if LIVE_AWS:
    try:
        response = dp.start_batch_evaluation(
            batchEvaluationName=f"EcommerceOptInsights{RUN_SUFFIX}",
            description="Section 05 optimization insights over production trace evidence",
            insights=[{"insightId": insight_id} for insight_id in INSIGHT_IDS],
            dataSourceConfig={
                "cloudWatchLogs": {
                    "serviceNames": BATCH_SERVICE_NAMES,
                    "logGroupNames": INSIGHT_LOG_GROUP_NAMES,
                    "filterConfig": {
                        "timeRange": {
                            "startTime": start_time,
                            "endTime": end_time,
                        }
                    },
                }
            },
            clientToken=str(uuid.uuid4()),
        )
        insights_result.update(
            {
                "status": "STARTED",
                "batch_evaluation_id": response.get("batchEvaluationId"),
                "batch_evaluation_arn": response.get("batchEvaluationArn"),
            }
        )
        print(f"Started insights job: {insights_result['batch_evaluation_id']}")
        final = {}
        for poll in range(1, 181):  # insights over real traces took 56 min for 5 sessions; budget 90 min
            final = dp.get_batch_evaluation(batchEvaluationId=insights_result["batch_evaluation_id"])
            status = final.get("status")
            print(f"  poll {poll:02d}: {status}")
            if status in TERMINAL_BATCH_STATUSES:
                break
            time.sleep(30)
        insights_result.update(
            {
                "status": final.get("status", insights_result["status"]),
                "statistics": final.get("statistics", {}),
                "evaluation_results": final.get("evaluationResults", {}),
                "error_details": final.get("errorDetails") or [],
                "error_count": len(final.get("errorDetails") or []),
                "has_failure_analysis": bool(final.get("failureAnalysisResult") or final.get("failureAnalysisOutput")),
                "has_user_intent": bool(final.get("userIntentResult") or final.get("userIntentOutput")),
                "has_execution_summary": bool(final.get("executionSummaryResult") or final.get("executionSummaryOutput")),
            }
        )
        insights_result["gate"] = validate_insights_gate(insights_result)
    except (ClientError, BotoCoreError) as exc:
        insights_result.update(
            {
                "status": "FAILED_API",
                "error_type": exc.__class__.__name__,
            }
        )
else:
    print("Live AWS calls skipped by SECTION05_SKIP_LIVE_AWS=1")

display(pd.DataFrame([insights_result]))


## Step 3: Generate Recommendation Candidates

This cell requests AgentCore recommendations for prompt and tool-description improvements.

The cell has three phases. First it creates recommendation-safe runtime traffic, using prompts that exercise the current agent without introducing policy text that can look like prompt-injection material. Then it waits until those traces are visible to CloudWatch and AgentCore indexing. Finally it starts the prompt and tool-description recommendation jobs. The tool-description recommendation uses the current endpoint service name for the fresh recommendation traffic and Gateway-qualified tool names because model and tool spans record tools in that shape.

Inspect `trace_visibility`, recommendation IDs, `has_candidate`, and the recommendation gate. The expected successful outcome is a completed recommendation job with usable candidate material: prompt text for a prompt candidate, and tool-description changes for observed tools. When a recommendation job fails or returns no candidate material, treat that as a blocking investigation result before creating treatment bundles. A failed recommendation is useful evidence, but it is not a treatment candidate.


In [ ]:

def poll_recommendation(recommendation_id: str, max_polls: int = 30) -> dict:
    terminal = {"COMPLETED", "FAILED"}
    last = {}
    for poll in range(1, max_polls + 1):
        last = dp.get_recommendation(recommendationId=recommendation_id)
        status = last.get("status")
        print(f"  recommendation poll {poll:02d}: {status}")
        if status in terminal:
            break
        time.sleep(30)
    return last

system_prompt_scope = "recommendation_safe_behavior_summary"
current_system_prompt = behavior.get("system_prompts", {}).get("customer") or combined_system_prompt(behavior)
recommendation_prompt_input = recommendation_safe_system_prompt(behavior)
current_tool_descriptions = behavior.get("tool_descriptions", {})
gateway_tool_descriptions = load_gateway_tool_descriptions()

recommendation_trace_sessions = []
recommendation_trace_start_time = start_time
recommendation_trace_end_time = end_time
if LIVE_AWS:
    recommendation_trace_start_time = datetime.now(timezone.utc) - timedelta(seconds=30)
    cognito_client = boto3.client("cognito-idp", region_name=REGION)
    runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)
    suffix = str(deployment.get("deployment_id", "")).split("-")[-1]
    user_pool_id = deployment["cognito"]["user_pool_id"]
    user_client_id = deployment["cognito"]["user_client_id"]
    test_password = os.environ.get("SECTION03_TEST_PASSWORD", f"{suffix}Aa1!z9")
    customer_email = os.environ.get("SECTION03_CUSTOMER_EMAIL", f"customer+{suffix}@example.com")
    admin_email = os.environ.get("SECTION03_ADMIN_EMAIL", f"admin+{suffix}@example.com")
    customer_tokens = get_user_token(cognito_client, user_pool_id, user_client_id, customer_email, test_password)
    admin_tokens = get_user_token(cognito_client, user_pool_id, user_client_id, admin_email, test_password)
    if not customer_tokens.get("id_token") or not admin_tokens.get("id_token"):
        raise RuntimeError("Could not authenticate demo users for recommendation trace traffic")

    recommendation_trace_prompts = [
        {
            "actor_role": "admin",
            "prompt": "Check inventory for PROD-001 and summarize the stock status.",
            "tokens": admin_tokens,
        },
        {
            "actor_role": "customer",
            "prompt": "Show product details for PROD-001.",
            "tokens": customer_tokens,
        },
    ]
    print("Generating safe recommendation trace traffic...")
    for item in recommendation_trace_prompts:
        session_id = f"s05-recommendation-{item['actor_role']}-{uuid.uuid4().hex}"
        result = invoke_agent_runtime(
            runtime_client,
            RUNTIME_ARN,
            session_id,
            {
                "prompt": item["prompt"],
                "bearer_token": item["tokens"].get("id_token", ""),
                "access_token": item["tokens"].get("access_token", ""),
                "session_id": session_id,
            },
        )
        recommendation_trace_sessions.append(
            {
                "session_id": session_id,
                "actor_role": item["actor_role"],
                "status": result.get("status", "error"),
                "tools_used": result.get("metadata", {}).get("tools_used", []),
            }
        )
        print(
            f"  {session_id}: status={recommendation_trace_sessions[-1]['status']} "
            f"tools={recommendation_trace_sessions[-1]['tools_used']}"
        )
        time.sleep(1)
    print(f"Authenticated users: customer={redact_email(customer_email)}, admin={redact_email(admin_email)}")

observed_from_recommendation_traffic = []
for item in recommendation_trace_sessions:
    for tool_name in item.get("tools_used") or []:
        if tool_name in current_tool_descriptions and tool_name not in observed_from_recommendation_traffic:
            observed_from_recommendation_traffic.append(tool_name)
observed_tools_for_recommendation = observed_from_recommendation_traffic or observed_tool_names(
    online_evidence, current_tool_descriptions, limit=1
)
observed_tools_for_recommendation = observed_tools_for_recommendation[:1]
tool_recommendation_inputs = build_tool_recommendation_inputs(
    gateway_tool_descriptions,
    observed_tools_for_recommendation,
)
tool_trace_service_names = RECOMMENDATION_SERVICE_NAMES

def wait_for_recommendation_trace_evidence(session_ids, required_tools, *, timeout_seconds=360, poll_seconds=20):
    if not session_ids:
        return {"status": "SKIPPED_NO_GENERATED_TRAFFIC"}
    logs_client = boto3.client("logs", region_name=REGION)
    deadline = time.time() + timeout_seconds
    required_markers = []
    for tool_name in required_tools:
        required_markers.extend([tool_name, f"ProductTools___{tool_name}"])
    last_summary = {"session_hits": 0, "tool_hits": 0}
    while time.time() < deadline:
        messages = []
        for log_group in LOG_GROUP_NAMES:
            for session_id in session_ids:
                try:
                    response = logs_client.filter_log_events(
                        logGroupName=log_group,
                        filterPattern=f'"{session_id}"',
                        startTime=int((recommendation_trace_start_time.timestamp() - 60) * 1000),
                        endTime=int((time.time() + 60) * 1000),
                        limit=100,
                    )
                except Exception as exc:
                    print(f"  Trace visibility warning for {log_group}: {exc.__class__.__name__}")
                    continue
                messages.extend(event.get("message", "") for event in response.get("events", []))
        session_hits = sum(1 for session_id in session_ids if any(session_id in message for message in messages))
        tool_hits = sum(1 for marker in required_markers if any(marker in message for message in messages))
        last_summary = {"session_hits": session_hits, "tool_hits": tool_hits, "messages_seen": len(messages)}
        if session_hits == len(session_ids) and tool_hits > 0:
            return {"status": "VISIBLE", **last_summary}
        print(f"  Waiting {poll_seconds}s for recommendation trace visibility: {last_summary}")
        time.sleep(poll_seconds)
    raise RuntimeError(f"Recommendation traces were not visible in time: {last_summary}")

trace_visibility = {"status": "SKIPPED"}
if LIVE_AWS:
    print("Waiting for generated recommendation traces to become queryable...")
    trace_visibility = wait_for_recommendation_trace_evidence(
        [item["session_id"] for item in recommendation_trace_sessions],
        observed_tools_for_recommendation,
    )
    recommendation_trace_end_time = datetime.now(timezone.utc)
    print(f"Recommendation trace visibility: {trace_visibility}")
    index_wait_seconds = int(os.environ.get("SECTION05_RECOMMENDATION_INDEX_WAIT_SECONDS", "240"))
    print(f"Waiting {index_wait_seconds} seconds for AgentCore recommendation indexing...")
    time.sleep(index_wait_seconds)
    recommendation_trace_end_time = datetime.now(timezone.utc)
prompt_agent_traces = {
    "cloudwatchLogs": {
        "logGroupArns": LOG_GROUP_ARNS,
        "serviceNames": RECOMMENDATION_SERVICE_NAMES,
        "startTime": recommendation_trace_start_time,
        "endTime": recommendation_trace_end_time,
    }
}
tool_agent_traces = {
    "cloudwatchLogs": {
        "logGroupArns": LOG_GROUP_ARNS,
        "serviceNames": tool_trace_service_names,
        "startTime": recommendation_trace_start_time,
        "endTime": recommendation_trace_end_time,
    }
}

recommendations = {
    "system_prompt": {
        "status": "SKIPPED" if not LIVE_AWS else "NOT_STARTED",
        "submitted_prompt_scope": system_prompt_scope,
        "prompt_input_mode": "recommendation_safe_summary",
        "has_candidate": False,
        "source": "cloudwatch_logs",
        "service_names": RECOMMENDATION_SERVICE_NAMES,
        "recommendation_trace_sessions": recommendation_trace_sessions,
        "trace_visibility": trace_visibility,
    },
    "tool_descriptions": {
        "status": "SKIPPED" if not LIVE_AWS else "NOT_STARTED",
        "has_candidate": False,
        "source": "cloudwatch_logs",
        "service_names": tool_trace_service_names,
        "tool_name_mode": "gateway_qualified",
        "submitted_tools": [item["toolName"] for item in tool_recommendation_inputs],
        "recommendation_trace_sessions": recommendation_trace_sessions,
        "trace_visibility": trace_visibility,
    },
    "feedback_signal_summary": feedback_signal_summary,
}
recommended_system_prompt = None
recommended_tool_descriptions = {}

if LIVE_AWS:
    try:
        sp_response = dp.start_recommendation(
            name=f"EcommerceSpRec{RUN_SUFFIX}",
            description="Section 05 system prompt recommendation from production traces",
            type="SYSTEM_PROMPT_RECOMMENDATION",
            recommendationConfig={
                "systemPromptRecommendationConfig": {
                    "systemPrompt": {"text": recommendation_prompt_input},
                    "agentTraces": prompt_agent_traces,
                    "evaluationConfig": {
                        "evaluators": [
                            {"evaluatorArn": "arn:aws:bedrock-agentcore:::evaluator/Builtin.GoalSuccessRate"}
                        ]
                    },
                }
            },
            clientToken=str(uuid.uuid4()),
        )
        sp_id = sp_response.get("recommendationId")
        print(f"Started system prompt recommendation: {sp_id}")
        sp_final = poll_recommendation(sp_id)
        recommended_system_prompt = extract_system_prompt_recommendation(sp_final)
        sp_error = (sp_final.get("recommendationResult") or {}).get("systemPromptRecommendationResult", {})
        recommendations["system_prompt"].update(
            {
                "status": sp_final.get("status"),
                "recommendation_id": sp_id,
                "has_candidate": bool(recommended_system_prompt),
                "text_changed": bool(recommended_system_prompt and recommended_system_prompt != current_system_prompt),
                "error_code": sp_error.get("errorCode"),
            }
        )
    except (ClientError, BotoCoreError) as exc:
        recommendations["system_prompt"].update(
            {
                "status": "FAILED_API",
                "has_candidate": False,
                "error_type": exc.__class__.__name__,
            }
        )

    try:
        td_response = dp.start_recommendation(
            name=f"EcommerceTdRec{RUN_SUFFIX}",
            description="Section 05 tool description recommendation from production traces",
            type="TOOL_DESCRIPTION_RECOMMENDATION",
            recommendationConfig={
                "toolDescriptionRecommendationConfig": {
                    "toolDescription": {
                        "toolDescriptionText": {
                            "tools": [
                                {
                                    "toolName": item["toolName"],
                                    "toolDescription": item["toolDescription"],
                                }
                                for item in tool_recommendation_inputs
                            ]
                        }
                    },
                    "agentTraces": tool_agent_traces,
                }
            },
            clientToken=str(uuid.uuid4()),
        )
        td_id = td_response.get("recommendationId")
        print(f"Started tool description recommendation: {td_id}")
        td_final = poll_recommendation(td_id)
        recommended_tool_descriptions = extract_tool_description_recommendations(td_final, current_tool_descriptions)
        td_error = (td_final.get("recommendationResult") or {}).get("toolDescriptionRecommendationResult", {})
        recommendations["tool_descriptions"].update(
            {
                "status": td_final.get("status"),
                "recommendation_id": td_id,
                "source": "cloudwatch_logs",
                "has_candidate": bool(recommended_tool_descriptions),
                "changed_tool_count": len(recommended_tool_descriptions),
                "error_code": td_error.get("errorCode"),
            }
        )
    except (ClientError, BotoCoreError) as exc:
        recommendations["tool_descriptions"].update(
            {
                "status": "FAILED_API",
                "has_candidate": False,
                "error_type": exc.__class__.__name__,
            }
        )

recommendation_rows = []
for name, result in recommendations.items():
    recommendation_rows.append({"candidate": name, **result})
display(pd.DataFrame(recommendation_rows))

try:
    recommendation_gate = validate_recommendation_gate(recommendations)
    recommendations["gate"] = recommendation_gate
except Exception as exc:
    recommendations["gate"] = {
        "status": "FAILED",
        "error": str(exc),
    }
    raise RuntimeError(str(exc)) from exc

if not recommended_system_prompt or not recommended_tool_descriptions:
    raise RuntimeError(
        "Recommendation gate passed without complete candidate material; "
        "inspect recommendation extraction before creating treatment bundles."
    )


## Step 4: Create Configuration-Bundle Candidates

This cell packages the current behavior and recommended behavior as auditable configuration bundles.

The control bundle should represent the deployed behavior source. The treatment bundle should represent a completed recommendation candidate. Bundle IDs and versions make the candidate reproducible, but creating a bundle does not send traffic to it.

Also inspect `bundle_readiness`. A config bundle only affects behavior when the deployed runtime reads `BedrockAgentCoreContext.get_config_bundle()` and applies the values before agent invocation. If the source code has the hook but the deployment manifest does not record `runtime.config_bundle_hook_version=1`, the correct result is blocked until Section 03 is redeployed.

In [ ]:

bundle_readiness = detect_config_bundle_readiness()
print(json.dumps(bundle_readiness, indent=2))

configuration_bundles = {
    "control": {"status": "SKIPPED" if not LIVE_AWS else "NOT_STARTED"},
    "treatment": {"status": "SKIPPED" if not LIVE_AWS else "NOT_STARTED"},
}

treatment_behavior = build_treatment_behavior(
    behavior,
    recommended_system_prompt=recommended_system_prompt,
    recommended_tool_descriptions=recommended_tool_descriptions,
)

if LIVE_AWS:
    try:
        control_response = ctrl.create_configuration_bundle(
            bundleName=f"EcommerceControl{RUN_SUFFIX}",
            description="Section 05 control bundle: current Product Catalog Agent behavior",
            components={
                RUNTIME_ARN: {
                    "configuration": build_bundle_configuration(behavior),
                }
            },
            commitMessage="Control: current prompt and tool descriptions from the Section 03 baseline",
            clientToken=str(uuid.uuid4()),
        )
        configuration_bundles["control"] = summarize_configuration_bundle_response(control_response)
    except (ClientError, BotoCoreError) as exc:
        configuration_bundles["control"] = {
            "status": "FAILED_API",
            "error_type": exc.__class__.__name__,
        }

    try:
        treatment_response = ctrl.create_configuration_bundle(
            bundleName=f"EcommerceTreatment{RUN_SUFFIX}",
            description="Section 05 treatment bundle: recommendation candidate from AgentCore/Section 04 evidence",
            components={
                RUNTIME_ARN: {
                    "configuration": build_bundle_configuration(treatment_behavior),
                }
            },
            commitMessage="Treatment: recommendation candidate; not promoted without experiment evidence",
            clientToken=str(uuid.uuid4()),
        )
        configuration_bundles["treatment"] = summarize_configuration_bundle_response(treatment_response)
    except (ClientError, BotoCoreError) as exc:
        configuration_bundles["treatment"] = {
            "status": "FAILED_API",
            "error_type": exc.__class__.__name__,
        }

bundle_rows = [{"variant": key, **value} for key, value in configuration_bundles.items()]
display(pd.DataFrame(bundle_rows))


## Step 5: Choose The Runnable Experiment Pattern

This cell decides and, when prerequisites are met, starts the experiment pattern that matches the candidate change.

Configuration-only changes use a config-bundle A/B pattern when the deployed runtime can read bundle values. The notebook first checks that Section 04 created a healthy online-evaluation evidence plane, then creates a fresh A/B-specific online evaluation config for this experiment. That fresh config watches the runtime service and the log groups used by the A/B Gateway traffic, using a short session window so evaluator metrics can appear during the workshop run.

Expect this step to take about 16-20 minutes in the workshop environment. Most of that time is waiting for Gateway traces, online evaluation scoring, and A/B result polling rather than sending the test requests.

The cell then creates a dedicated AgentCore Gateway target for the runtime, enables Gateway trace delivery, and fails fast if trace delivery is only partial. It starts the A/B test with AgentCore variant names `C` and `T1`, and sends signed traffic only after the A/B test is `ACTIVE` and `RUNNING`. Before polling for results, it checks for the minimum scoreability signals: at least one model-span session, at least one runtime configuration-bundle fetch, no MCP 400 evidence, and no bundle access-denied evidence. A successful HTTP response alone is not enough evidence for A/B scoring.

`READY_NOT_STARTED` means the prerequisites exist but the operator disabled live A/B for this run. `RUNNING_NO_RESULTS_YET` means clean traffic was routed and the A/B test was intentionally left running for more metrics. `TRAFFIC_ROUTED_NO_RESULTS_STOPPED` means clean traffic was routed, but the notebook stopped the test before evaluator metrics were ready. `NO_WINNER`, `REGRESSION`, and `COMPLETED_WITH_WINNER` are metric-based outcomes. `ERROR` means the service reported A/B errors, and promotion is not allowed. `BLOCKED` means a prerequisite is missing and promotion is not allowed.

Code, tool, runtime, or Gateway changes use a target-based canary pattern. This notebook records that pattern as a dry-run plan unless a real v2 runtime or Gateway target exists with Section 03-style pre-production gate evidence. For a real canary, inspect the baseline runtime ID, candidate runtime or target ID, CI/pre-production gate status, traffic split, online evaluator scores, rollback target, and promotion eligibility.


In [ ]:
planned_variants = [
    {
        "name": "C",
        "label": "control",
        "weight": 50,
        "bundle_id": configuration_bundles.get("control", {}).get("bundle_id"),
        "bundle_version": configuration_bundles.get("control", {}).get("version_id"),
    },
    {
        "name": "T1",
        "label": "treatment",
        "weight": 50,
        "bundle_id": configuration_bundles.get("treatment", {}).get("bundle_id"),
        "bundle_version": configuration_bundles.get("treatment", {}).get("version_id"),
    },
]

if not bundle_readiness.get("supported"):
    config_bundle_ab = blocked_experiment(
        experiment_type="config_bundle_ab_test",
        reason="BLOCKED_CONFIG_BUNDLE_HOOK_MISSING",
        prerequisites=[
            "Runtime source must import BedrockAgentCoreContext",
            "Runtime source must call BedrockAgentCoreContext.get_config_bundle()",
            "Runtime must apply bundle system prompt and tool description values before agent invocation",
            "Section 03 must redeploy the runtime and save runtime.config_bundle_hook_version=1 in deployment_manifest.json",
        ],
        planned_variants=planned_variants,
    )
elif not all(item.get("bundle_id") and item.get("bundle_version") for item in planned_variants):
    config_bundle_ab = blocked_experiment(
        experiment_type="config_bundle_ab_test",
        reason="BLOCKED_CONFIGURATION_BUNDLES_NOT_READY",
        prerequisites=["Control and treatment bundles must both be created"],
        planned_variants=planned_variants,
    )
else:
    online_eval = online_evidence.get("online_evaluation", {})
    source_online_eval_arn = online_eval.get("online_evaluation_config_arn")
    source_online_eval_id = online_eval.get("online_evaluation_config_id")
    enable_ab_test = os.environ.get("SECTION05_ENABLE_AB_TEST", "1") == "1"
    if not source_online_eval_arn or not source_online_eval_id:
        config_bundle_ab = blocked_experiment(
            experiment_type="config_bundle_ab_test",
            reason="BLOCKED_ONLINE_EVALUATION_CONFIG_MISSING",
            prerequisites=["Section 04 online evaluation config ARN must be available"],
            planned_variants=planned_variants,
        )
    elif not enable_ab_test:
        config_bundle_ab = {
            "type": "config_bundle_ab_test",
            "status": "READY_NOT_STARTED",
            "reason": "Set SECTION05_ENABLE_AB_TEST=1 to start live A/B traffic",
            "online_evaluation_config_arn": source_online_eval_arn,
            "planned_variants": planned_variants,
            "promotion_allowed": False,
        }
    else:
        iam_client = boto3.client("iam", region_name=REGION)
        logs_client = boto3.client("logs", region_name=REGION)
        xray_client = boto3.client("xray", region_name=REGION)

        live_online_eval = ctrl.get_online_evaluation_config(onlineEvaluationConfigId=source_online_eval_id)
        alignment_result = ensure_online_evaluation_alignment(
            ctrl,
            live_online_eval,
            required_service_names=SERVICE_NAMES,
            required_log_group_names=LOG_GROUP_NAMES,
        )
        live_online_eval = alignment_result["config"]
        require_online_evaluation_ready(
            live_online_eval,
            required_service_names=SERVICE_NAMES,
            required_log_group_names=LOG_GROUP_NAMES,
        )
        source_online_eval_summary = summarize_online_evaluation_config(live_online_eval)
        source_online_eval_summary["alignment"] = alignment_result["alignment"]
        source_online_eval_summary["alignment_updated"] = alignment_result["updated"]
        source_online_eval_summary["source"] = "section04_prerequisite"

        evaluation_role_arn = live_online_eval.get("evaluationExecutionRoleArn")
        if not evaluation_role_arn:
            evaluation_role_name = os.environ.get("SECTION04_EVALUATION_ROLE_NAME", "ecommerce-workshop-evaluation-role")
            evaluation_role_arn = iam_client.get_role(RoleName=evaluation_role_name)["Role"]["Arn"]

        ab_role_name = os.environ.get("SECTION05_AB_GATEWAY_ROLE_NAME", "ecommerce-workshop-ab-gateway-role")
        ab_role = ensure_ab_gateway_role(
            iam_client,
            role_name=ab_role_name,
            runtime_arn=RUNTIME_ARN,
        )
        ab_role_arn = ab_role["Arn"]
        ab_target_name = f"ProductRuntime{RUN_SUFFIX}"
        ab_gateway = create_runtime_gateway_and_target(
            ctrl,
            gateway_name=f"ecommerce-workshop-ab-{RUN_SUFFIX}",
            target_name=ab_target_name,
            role_arn=ab_role_arn,
            runtime_arn=RUNTIME_ARN,
            region=REGION,
            account_id=ACCOUNT_ID,
        )
        gateway_trace_delivery = ensure_gateway_trace_delivery(
            logs_client,
            xray_client,
            gateway_arn=ab_gateway["gateway_arn"],
            delivery_source_name=f"section05-ab-{RUN_SUFFIX}-traces",
        )
        require_gateway_trace_delivery_ready(gateway_trace_delivery)

        ab_online_eval_summary = create_ab_online_evaluation_config(
            ctrl,
            name=f"EcommerceABEval{RUN_SUFFIX}",
            description="Section 05 A/B-specific online evaluation for config-bundle scoring",
            log_group_names=LOG_GROUP_NAMES,
            service_names=SERVICE_NAMES,
            evaluation_execution_role_arn=evaluation_role_arn,
            evaluator_ids=("Builtin.GoalSuccessRate", "Builtin.Helpfulness"),
            session_timeout_minutes=int(os.environ.get("SECTION05_AB_SESSION_TIMEOUT_MINUTES", "1")),
        )
        ab_online_eval_live = ctrl.get_online_evaluation_config(
            onlineEvaluationConfigId=ab_online_eval_summary["online_evaluation_config_id"]
        )
        require_online_evaluation_ready(
            ab_online_eval_live,
            required_service_names=SERVICE_NAMES,
            required_log_group_names=LOG_GROUP_NAMES,
        )
        ab_online_eval_summary = summarize_online_evaluation_config(ab_online_eval_live)
        ab_online_eval_summary["source"] = "section05_ab_specific"
        ab_online_eval_summary["session_timeout_minutes"] = int(os.environ.get("SECTION05_AB_SESSION_TIMEOUT_MINUTES", "1"))
        online_eval_arn = ab_online_eval_summary["online_evaluation_config_arn"]

        ab_response = create_config_bundle_ab_test(
            dp,
            name=f"EcommerceBundleAB{RUN_SUFFIX}",
            gateway_arn=ab_gateway["gateway_arn"],
            role_arn=ab_role_arn,
            online_evaluation_config_arn=online_eval_arn,
            control_bundle=configuration_bundles["control"],
            treatment_bundle=configuration_bundles["treatment"],
        )
        ab_test_id = ab_response["abTestId"]
        ab_running = wait_for_ab_running(dp, ab_test_id)
        require_ab_running(ab_running)

        suffix = str(deployment.get("deployment_id", "")).split("-")[-1]
        cognito_client = boto3.client("cognito-idp", region_name=REGION)
        test_password = os.environ.get("SECTION03_TEST_PASSWORD", f"{suffix}Aa1!z9")
        customer_email = os.environ.get("SECTION03_CUSTOMER_EMAIL", f"customer+{suffix}@example.com")
        admin_email = os.environ.get("SECTION03_ADMIN_EMAIL", f"admin+{suffix}@example.com")
        customer_tokens = get_user_token(cognito_client, deployment["cognito"]["user_pool_id"], deployment["cognito"]["user_client_id"], customer_email, test_password)
        admin_tokens = get_user_token(cognito_client, deployment["cognito"]["user_pool_id"], deployment["cognito"]["user_client_id"], admin_email, test_password)
        ab_payload_specs = [
            ("customer", "details", "Show product details for PROD-001.", customer_tokens),
            ("customer", "compare", "Compare PROD-001 and PROD-008 for battery life and portability.", customer_tokens),
            ("customer", "return_policy", "What is the return policy for audio products?", customer_tokens),
            ("customer", "recommend", "Recommend audio accessories under $80.", customer_tokens),
            ("customer", "inventory_customer", "Is PROD-001 currently in stock?", customer_tokens),
            ("customer", "category_search", "Find audio products under $150 with strong customer ratings.", customer_tokens),
            ("customer", "warranty", "Explain warranty coverage for PROD-001 in plain English.", customer_tokens),
            ("admin", "inventory", "Check inventory for PROD-001 and summarize the current stock status.", admin_tokens),
            ("admin", "pricing_read", "Look up PROD-008 and summarize whether the current price looks discounted.", admin_tokens),
            ("admin", "admin_compare", "Compare PROD-001 and PROD-015 for an operations review.", admin_tokens),
        ]
        ab_payloads = []
        for actor_role, prompt_name, prompt_text, tokens in ab_payload_specs:
            session_id = f"s05-ab-{actor_role}-{uuid.uuid4().hex}"
            ab_payloads.append(
                {
                    "session_id": session_id,
                    "actor_role": actor_role,
                    "prompt_name": prompt_name,
                    "payload": {
                        "prompt": prompt_text,
                        "bearer_token": tokens.get("id_token", ""),
                        "access_token": tokens.get("access_token", ""),
                        "session_id": session_id,
                    },
                }
            )
        traffic = invoke_ab_gateway(
            gateway_url=ab_gateway["gateway_url"],
            target_name=ab_target_name,
            region=REGION,
            payloads=ab_payloads,
        )
        require_gateway_traffic_success(traffic, expected_count=len(ab_payloads))

        trace_wait_seconds = int(os.environ.get("SECTION05_AB_TRACE_TIMEOUT_SECONDS", "420"))
        trace_poll_seconds = int(os.environ.get("SECTION05_AB_TRACE_POLL_SECONDS", "20"))
        ab_trace_evidence = wait_for_ab_trace_evidence(
            logs_client,
            sessions=traffic["sessions"],
            log_group_names=LOG_GROUP_NAMES,
            timeout_seconds=trace_wait_seconds,
            poll_seconds=trace_poll_seconds,
        )
        require_ab_trace_evidence(ab_trace_evidence)

        poll_count = int(os.environ.get("SECTION05_AB_RESULT_POLLS", "25"))
        poll_seconds = int(os.environ.get("SECTION05_AB_RESULT_POLL_SECONDS", "60"))
        ab_final = poll_ab_results(dp, ab_test_id, max_polls=poll_count, poll_seconds=poll_seconds)
        ab_summary = summarize_ab_test(ab_final)
        keep_ab_running = os.environ.get("SECTION05_KEEP_AB_TEST_RUNNING", "0") == "1"
        finalize_ab_for_decision = (
            os.environ.get("SECTION05_FINALIZE_AB_TEST_FOR_DECISION", "1") == "1"
            and not keep_ab_running
        )
        finalization = {
            "stop_before_decision": finalize_ab_for_decision,
            "stopped_by_notebook": False,
            "reason": "Initial polling already produced decision metrics" if ab_summary.get("has_results") else "Not finalized in experiment step",
        }
        if finalize_ab_for_decision and not ab_summary.get("service_error"):
            post_stop_polls = int(os.environ.get("SECTION05_POST_STOP_RESULT_POLLS", "15"))
            post_stop_poll_seconds = int(os.environ.get("SECTION05_POST_STOP_RESULT_POLL_SECONDS", "60"))
            finalization_result = finalize_ab_test_for_decision(
                dp,
                ab_test_id,
                stop_before_decision=True,
                post_stop_polls=post_stop_polls,
                poll_seconds=post_stop_poll_seconds,
            )
            ab_summary = finalization_result["summary"]
            finalization = finalization_result["finalization"]
        ab_status = derive_config_bundle_ab_status(
            ab_summary,
            keep_ab_running=ab_summary.get("execution_status") == "RUNNING",
        )
        config_bundle_ab = {
            "type": "config_bundle_ab_test",
            "status": ab_status,
            "reason": "Live config-bundle A/B traffic was routed through AgentCore Gateway",
            "gateway": ab_gateway,
            "gateway_trace_delivery": gateway_trace_delivery,
            "online_evaluation_prerequisite": source_online_eval_summary,
            "online_evaluation": ab_online_eval_summary,
            "ab_running": summarize_ab_test(ab_running),
            "ab_test": ab_summary,
            "finalization": finalization,
            "traffic": traffic,
            "trace_evidence": ab_trace_evidence,
            "online_evaluation_config_arn": online_eval_arn,
            "planned_variants": planned_variants,
            "promotion_allowed": False,
        }

target_canary = dry_run_target_canary_plan(
    baseline_runtime=RUNTIME,
    candidate_runtime=None,
    ci_gate=None,
)

experiments = {
    "config_bundle_ab_test": config_bundle_ab,
    "target_based_canary": target_canary,
}

display(pd.DataFrame([
    {"experiment": "config bundle A/B", "status": config_bundle_ab.get("status"), "reason": config_bundle_ab.get("reason")},
    {"experiment": "target canary", "status": target_canary.get("status"), "reason": target_canary.get("reason")},
]))


## Step 6: Interpret A/B Metrics And Choose The Promotion Path

This cell turns raw A/B service output into a release decision.

A successful Gateway call only proves that traffic reached the active A/B Gateway. It does not prove that the treatment won. The decision-ready evidence is the A/B evaluator table: for each evaluator, compare the control mean with the treatment mean, percent change, p-value or significance flag, and sample size. The promotion rule is intentionally conservative: do not promote when metrics are missing, service errors exist, sample size is too small, or any treatment evaluator regresses. A treatment becomes promotion-ready only when evaluator metrics show a significant treatment improvement with no significant regression, and the operator still makes the final decision explicitly.

Inspect the metric table, `metric_review.status`, `recommended_release_decision`, `promotion_candidate_ready`, and `next_action`. For a workshop decision run, the notebook stops the A/B test before interpreting final metrics unless `SECTION05_KEEP_AB_TEST_RUNNING=1` is set. If the status is `NO_RESULTS_AFTER_STOP`, the service did not produce evaluator metrics even after finalization, so do not promote; rerun with more representative traffic or investigate the online evaluation and A/B scoring configuration. If the status is `COMPLETED_WITH_WINNER`, review the metrics first. To record approval for the already-reviewed in-memory result, set `SECTION05_OPERATOR_DECISION=promote` before rerunning only the final artifact-writing cells; rerunning the full notebook creates a new experiment.


In [ ]:

decision_poll_count = int(os.environ.get("SECTION05_DECISION_RESULT_POLLS", "1"))
decision_poll_seconds = int(os.environ.get("SECTION05_DECISION_RESULT_POLL_SECONDS", "60"))
min_variant_sample_size = int(os.environ.get("SECTION05_MIN_VARIANT_SAMPLE_SIZE", "1"))
finalize_before_decision = os.environ.get("SECTION05_FINALIZE_AB_TEST_FOR_DECISION", "1") == "1"

ab_test_id_for_review = (config_bundle_ab.get("ab_test") or {}).get("ab_test_id")
if LIVE_AWS and ab_test_id_for_review and config_bundle_ab.get("status") == "RUNNING_NO_RESULTS_YET":
    print(
        "Finalizing A/B result before release decision "
        f"(stop={finalize_before_decision}, polls={decision_poll_count}, seconds={decision_poll_seconds})..."
    )
    finalization_result = finalize_ab_test_for_decision(
        dp,
        ab_test_id_for_review,
        stop_before_decision=finalize_before_decision,
        post_stop_polls=decision_poll_count,
        poll_seconds=decision_poll_seconds,
    )
    refreshed_summary = finalization_result["summary"]
    config_bundle_ab["ab_test"] = refreshed_summary
    config_bundle_ab["finalization"] = finalization_result["finalization"]
    config_bundle_ab["status"] = derive_config_bundle_ab_status(
        refreshed_summary,
        keep_ab_running=refreshed_summary.get("execution_status") == "RUNNING",
    )
    config_bundle_ab["decision_poll"] = {
        "poll_count": decision_poll_count,
        "poll_seconds": decision_poll_seconds,
        "refreshed_status": config_bundle_ab["status"],
        "has_results": refreshed_summary.get("has_results"),
        "metric_count": refreshed_summary.get("metric_count"),
    }
elif LIVE_AWS and ab_test_id_for_review and config_bundle_ab.get("status") == "TRAFFIC_ROUTED_NO_RESULTS_STOPPED":
    print("A/B test is already stopped with no evaluator metrics. Reading final state once more before decision...")
    refreshed_summary = summarize_ab_test(dp.get_ab_test(abTestId=ab_test_id_for_review))
    config_bundle_ab["ab_test"] = refreshed_summary
    config_bundle_ab["status"] = derive_config_bundle_ab_status(refreshed_summary, keep_ab_running=False)
else:
    print("Using the A/B summary already produced by the experiment step.")

ab_metric_review = build_ab_metric_review(
    config_bundle_ab.get("ab_test") or {},
    min_variant_sample_size=min_variant_sample_size,
)
config_bundle_ab["metric_review"] = ab_metric_review
experiments["config_bundle_ab_test"] = config_bundle_ab

metric_rows = ab_metric_review.get("metric_rows") or []
metric_columns = [
    "evaluator_id",
    "variant_name",
    "control_mean",
    "treatment_mean",
    "percent_change",
    "p_value",
    "is_significant",
    "control_sample_size",
    "treatment_sample_size",
    "meets_min_sample_size",
    "signal",
]
if metric_rows:
    display(pd.DataFrame(metric_rows)[metric_columns])
else:
    print("No evaluator metrics are available yet. Do not promote from this run.")

display(pd.DataFrame([
    {
        "metric_review_status": ab_metric_review.get("status"),
        "recommended_release_decision": ab_metric_review.get("recommended_release_decision"),
        "promotion_candidate_ready": ab_metric_review.get("promotion_candidate_ready"),
        "promotion_candidate": ab_metric_review.get("promotion_candidate"),
        "min_variant_sample_size": ab_metric_review.get("min_variant_sample_size"),
        "rationale": ab_metric_review.get("rationale"),
        "next_action": ab_metric_review.get("next_action"),
    }
]))

print("Decision rules:")
for rule in ab_metric_review.get("decision_rules", []):
    print(f"- {rule}")


## Step 7: Write Optimization And Release Artifacts

This cell writes the optimization manifest, release decision report, and promotion manifest from the interpreted A/B metric review.

These artifacts turn the notebook run into an auditable decision record: what evidence was loaded, what recommendations ran, what bundles were created, what experiment was possible, what decision was made, and what rollback or monitoring handoff would be used next. Inspect `config_bundle_ab.status`, `gateway_trace_delivery`, `trace_evidence`, `traffic`, `ab_test.has_results`, `ab_test.evaluator_metrics`, `promotion_allowed`, and `promotion_executed`. Those fields tell you whether signed traffic flowed through the active A/B Gateway, whether evaluator metrics are ready, whether a winner or regression was detected, and whether the run stopped before production change. Treatment exposure is only proven by the A/B service's variant/evaluator results, not by a successful Gateway invocation alone.

The release decision has four values. `promote` means the evidence supports a selected treatment, but a human still controls production change. `continue` means keep collecting evidence or review the result before changing production. `investigate` means a regression, service error, or incomplete experiment must be understood first. `rollback` is reserved for a case where the candidate has already affected production and must be reversed.

For this workshop, config-bundle promotion is recorded as a manual production-change plan. The notebook can prove a treatment is promotion-ready, but it does not switch the normal production Gateway/default behavior by itself. Therefore `SKIPPED_NO_PROMOTION`, `DRY_RUN_READY`, or `PRODUCTION_CHANGE_PLAN_ACKNOWLEDGED` can be correct final states, depending on the evidence and operator environment variables. `promotion_executed=true` is only valid when an artifact proves production traffic or default behavior was actually switched.


In [ ]:

ab_status = config_bundle_ab.get("status")
ab_metric_review = config_bundle_ab.get("metric_review", {})
promotion_candidate_ready = ab_metric_review.get("promotion_candidate_ready") is True
operator_decision = os.environ.get("SECTION05_OPERATOR_DECISION", "continue").lower()
execute_promotion = os.environ.get("SECTION05_EXECUTE_PROMOTION", "0") == "1"

if ab_status == "COMPLETED_WITH_WINNER" and promotion_candidate_ready and operator_decision == "promote":
    config_bundle_ab["promotion_allowed"] = True
    config_bundle_ab["operator_decision"] = "promote"
else:
    config_bundle_ab["operator_decision"] = operator_decision
experiments["config_bundle_ab_test"] = config_bundle_ab

optimization_manifest = build_optimization_manifest(
    evidence=evidence,
    behavior=behavior,
    insights=insights_result,
    recommendations=recommendations,
    bundle_readiness=bundle_readiness,
    configuration_bundles=configuration_bundles,
    experiments=experiments,
)
save_json(optimization_manifest, OPTIMIZATION_MANIFEST_PATH)

residual_risks = [
    "A code/runtime treatment needs a Section 03-style pre-production gate before target canary traffic.",
    f"Section 04 promoted runtime-error examples requiring review: {feedback_signal_summary['promoted_runtime_error_example_count']}.",
]
next_steps = [
    "After any promotion, hand monitoring back to Section 04 online evidence and feedback mining.",
]

if ab_status == "COMPLETED_WITH_WINNER":
    if not promotion_candidate_ready:
        decision = "continue"
        rationale = ab_metric_review.get("rationale") or "A/B metrics are present, but the promotion checklist is not satisfied."
        residual_risks.append("A/B metrics did not satisfy the promotion checklist, so the treatment is not promotion-ready.")
        next_steps.insert(0, ab_metric_review.get("next_action") or "Review A/B metrics before choosing a promotion action.")
    elif operator_decision == "promote":
        decision = "promote"
        rationale = "Config-bundle A/B metric review produced a promotion-ready treatment and the operator selected promote."
        next_steps.insert(0, "Use SECTION05_EXECUTE_PROMOTION=1 only to acknowledge the manual production-change plan; the notebook does not switch production traffic.")
    else:
        decision = "continue"
        rationale = "Config-bundle A/B produced a promotion-ready treatment; review the metric table before selecting an explicit promotion decision."
        residual_risks.append("Treatment traffic was routed, but promotion still needs an explicit human decision after metric review.")
        next_steps.insert(0, "Set SECTION05_OPERATOR_DECISION=promote only after reviewing A/B evaluator metrics and the promotion checklist.")
elif ab_status == "NO_WINNER":
    decision = "continue"
    rationale = "Config-bundle A/B produced evaluator results, but no statistically significant winner was found."
    residual_risks.append("Treatment traffic was routed, but the A/B result did not identify a promotion candidate.")
    next_steps.insert(0, "Continue collecting traffic or revise the treatment before another experiment.")
elif ab_status == "REGRESSION":
    decision = "investigate"
    rationale = "Config-bundle A/B produced evaluator results with a significant treatment regression."
    residual_risks.append("Treatment traffic showed a regression; do not promote this configuration bundle.")
    next_steps.insert(0, "Inspect A/B evaluator metrics, keep the control bundle active, and revise the treatment candidate.")
elif ab_status == "RUNNING_NO_RESULTS_YET":
    decision = "continue"
    rationale = "Config-bundle A/B routed traffic, but evaluator results were not available within the notebook polling window."
    residual_risks.append("Live A/B traffic was routed, but evaluator results were not available during this run.")
    next_steps.insert(0, "Re-open the A/B test status or rerun polling before making a promotion decision.")
elif ab_status == "TRAFFIC_ROUTED_NO_RESULTS_STOPPED":
    decision = "continue"
    rationale = "Config-bundle A/B routed clean traffic and was stopped, but evaluator results were not available before stop."
    residual_risks.append("Live A/B traffic was routed, but no evaluator metrics were available before the notebook stopped the test.")
    next_steps.insert(0, "Rerun with SECTION05_KEEP_AB_TEST_RUNNING=1 or increase the A/B polling window before deciding on promotion.")
elif ab_status == "ERROR":
    decision = "investigate"
    rationale = "Config-bundle A/B reported service-side errors; do not promote this treatment until the error details are understood."
    residual_risks.append("The A/B service reported errors, so evaluator metrics and treatment exposure are not clean validation evidence.")
    next_steps.insert(0, "Inspect optimization_manifest.json ab_test.error_details and rerun the experiment only after the service error is resolved.")
elif ab_status == "READY_NOT_STARTED":
    decision = "continue"
    rationale = "Configuration-bundle A/B is ready to start after an explicit operator decision."
    residual_risks.append("No live A/B traffic was routed because the operator disabled the A/B start for this run.")
    next_steps.insert(0, "Set SECTION05_ENABLE_AB_TEST=1 when ready to start live config-bundle A/B traffic.")
elif config_bundle_ab.get("reason") == "BLOCKED_CONFIG_BUNDLE_HOOK_MISSING":
    decision = "continue"
    rationale = "The treatment candidate is packaged, but the runtime does not yet consume configuration bundles, so live config-bundle A/B and promotion are blocked."
    residual_risks.append("No live treatment traffic was routed because the deployed runtime is not proven to consume configuration bundles.")
    residual_risks.append("The deployed runtime must be updated before config-bundle variants can affect behavior.")
    next_steps.insert(0, "Rerun Section 03 so the deployed manifest records runtime.config_bundle_hook_version=1.")
else:
    decision = "investigate"
    rationale = "Optimization evidence or configuration-bundle candidates are incomplete; do not promote."
    residual_risks.append("Experiment evidence is incomplete or blocked; promotion is not allowed.")
    next_steps.insert(0, "Inspect optimization_manifest.json for the blocked or failed experiment status before rerunning.")

release_decision_report = build_release_decision_report(
    optimization_manifest=optimization_manifest,
    decision=decision,
    rationale=rationale,
    residual_risks=residual_risks,
    next_steps=next_steps,
)
save_json(release_decision_report, RELEASE_DECISION_REPORT_PATH)

promotion_execution = {}
if release_decision_report.get("promotion_allowed") and execute_promotion:
    promotion_execution = {
        "status": "PRODUCTION_CHANGE_PLAN_ACKNOWLEDGED",
        "promotion_executed": False,
        "execution_mode": "explicit_operator_acknowledged_manual_change_required",
        "execution_result": {
            "production_traffic_switched": False,
            "reason": (
                "The workshop records the promotion-ready treatment and required manual "
                "production change. It does not switch the normal production Gateway or "
                "default runtime behavior automatically."
            ),
            "selected_treatment_bundle_id": configuration_bundles.get("treatment", {}).get("bundle_id"),
            "selected_treatment_bundle_version": configuration_bundles.get("treatment", {}).get("version_id"),
            "ab_test_id": (config_bundle_ab.get("ab_test") or {}).get("ab_test_id"),
            "required_actions": [
                "Review A/B metric_review.metric_rows and residual risks.",
                "Apply the selected bundle through the production routing/default-bundle mechanism used by the application.",
                "Record the production route/default state and rollback target after the switch.",
                "Return monitoring to Section 04 after the production change.",
            ],
        },
    }
elif release_decision_report.get("promotion_allowed"):
    promotion_execution = {
        "status": "DRY_RUN_READY",
        "promotion_executed": False,
        "execution_mode": "manual_approval_required",
        "execution_result": {
            "required_env": "SECTION05_EXECUTE_PROMOTION=1",
            "production_traffic_switched": False,
            "selected_treatment_bundle_id": configuration_bundles.get("treatment", {}).get("bundle_id"),
            "selected_treatment_bundle_version": configuration_bundles.get("treatment", {}).get("version_id"),
            "next_action": "A human must apply the production routing/default-bundle change outside this notebook.",
        },
    }

promotion_manifest = build_promotion_manifest(
    deployment=deployment,
    release_decision=release_decision_report,
    selected_variant={
        "candidate": "treatment",
        "config_bundle": configuration_bundles.get("treatment", {}),
        "experiment_status": config_bundle_ab.get("status"),
        "metric_review": ab_metric_review,
    },
    promotion_execution=promotion_execution,
)
save_json(promotion_manifest, PROMOTION_MANIFEST_PATH)

section05_timing = {
    "version": "1.0",
    "artifact_type": "section_05_timing",
    "created_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    "duration_seconds": round(time.time() - STARTED_AT, 2),
    "live_aws_calls_enabled": LIVE_AWS,
    "notebook": "05-agentcore-optimization/05-agentcore-optimization-experiments.ipynb",
}
save_json(section05_timing, SECTION_DIR / "section05_timing.json")

print(f"Wrote {OPTIMIZATION_MANIFEST_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {RELEASE_DECISION_REPORT_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {PROMOTION_MANIFEST_PATH.relative_to(REPO_ROOT)}")
print(f"Wrote {(SECTION_DIR / 'section05_timing.json').relative_to(REPO_ROOT)}")


## Step 8: Review The Release State

This cell prints the final state of the optimization run.

Use it as the operator summary. Check whether recommendations completed, whether an experiment was runnable, whether A/B traffic reached the Gateway, whether evaluator metrics are ready, whether promotion was allowed by evidence, and whether any production action remains manual. The correct next step depends on these statuses, not on the notebook simply reaching the last cell.

Expected clean endings are precise. `continue` with `SKIPPED_NO_PROMOTION` means no evidence-backed production change was requested. `continue` with `RUNNING_NO_RESULTS_YET` means traffic was routed but metrics still need time. `promote` with `DRY_RUN_READY` means evidence supports a treatment and the operator has not acknowledged the manual production change. `promote` with `PRODUCTION_CHANGE_PLAN_ACKNOWLEDGED` means the operator acknowledged the handoff, but the notebook still did not switch production traffic.


In [ ]:

review_rows = [
    {"artifact": "optimization manifest", "path": str(OPTIMIZATION_MANIFEST_PATH.relative_to(REPO_ROOT)), "status": optimization_manifest.get("experiments", {}).get("config_bundle_ab_test", {}).get("status")},
    {"artifact": "A/B metric review", "path": "optimization_manifest.json:experiments.config_bundle_ab_test.metric_review", "status": ab_metric_review.get("status")},
    {"artifact": "release decision", "path": str(RELEASE_DECISION_REPORT_PATH.relative_to(REPO_ROOT)), "status": release_decision_report.get("decision")},
    {"artifact": "promotion manifest", "path": str(PROMOTION_MANIFEST_PATH.relative_to(REPO_ROOT)), "status": promotion_manifest.get("status")},
    {"artifact": "timing", "path": "05-agentcore-optimization/section05_timing.json", "status": f"{section05_timing['duration_seconds']}s"},
]
display(pd.DataFrame(review_rows))
print(json.dumps({
    "decision": release_decision_report["decision"],
    "promotion_allowed": release_decision_report["promotion_allowed"],
    "promotion_executed": promotion_manifest["promotion_executed"],
    "ab_metric_review": ab_metric_review,
    "config_bundle_readiness": bundle_readiness,
}, indent=2))
